# Guide 6: Next-token prediction: from probabilities to text

*How LLMs work* · [Read the guide](https://www.vinaygautam.com/blog/how-llms-work/06-next-token) · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drvgautam/primer-on-aiengineering/blob/main/code/notebooks/06-next-token.ipynb)

This notebook holds every code block from Guide 6, in the guide's order: 11 code cells, 10 of which print an output the guide quotes. Each cell is preceded by the guide paragraph that introduces it, so you can keep the guide open beside the notebook. The saved outputs are the guide's; `tools/verify_notebooks.py` checks they still match.

Run the setup cell first, then the cells in order: like the guide's blocks, they share variables.

**Sections of the guide**

- From the last vector to a prediction — no code
- Training: a prediction at every position — 3 code cells
- Generation: one token at a time — 1 code cell
- Choosing a token: greedy decoding — 1 code cell
- Sampling and temperature — 3 code cells
- Cutting off the tail: top-k and top-p — 2 code cells
- The running example — 1 code cell
- What an output probability is not — no code

In [1]:
# Setup: run this cell first. It makes the repository's code/ folder the working directory, so
# every block below runs exactly as printed in the guide ("run from the repository root").
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/drvgautam/primer-on-aiengineering"
if "google.colab" in sys.modules and not any((p / "code" / "notebooks").is_dir() for p in [Path.cwd(), *Path.cwd().parents]):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/primer-on-aiengineering"], check=True)
    os.chdir("/content/primer-on-aiengineering")
    pins = [l.strip() for l in open("code/requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins,
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu"], check=True)

here = Path.cwd().resolve()
code_dir = next(p / "code" for p in [here, *here.parents] if (p / "code" / "notebooks").is_dir())
os.chdir(code_dir)

# The guide's outputs were produced with these versions, on CPU. Others may differ in the last digits.
import numpy, tiktoken, torch
pins = dict(l.strip().split("==") for l in open("requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch"))
for name, mod in (("numpy", numpy), ("tiktoken", tiktoken), ("torch", torch)):
    have = mod.__version__.split("+")[0]
    note = "" if have == pins[name] else f"  <- the guide used {pins[name]}; outputs may differ slightly (on Colab, restart the runtime and rerun this cell)"
    print(f"{name:9} {have}{note}")
print("working directory:", Path(code_dir.parent.name) / "code")

# %%gpt2 marks blocks the guide shows but doesn't run, because they need GPT-2 small's 548 MB weights.
# To run them, download the weights as series/running-example.py explains, then set:
#     os.environ["PRIMER_RUN_GPT2"] = "1"
from IPython.core.magic import register_cell_magic

@register_cell_magic
def gpt2(line, cell):
    if os.environ.get("PRIMER_RUN_GPT2") == "1":
        get_ipython().run_cell(cell)
    else:
        print("Skipped: needs GPT-2 weights (see the setup cell to enable).")


numpy     2.2.6
tiktoken  0.14.0
torch     2.14.1
working directory: primer-on-aiengineering/code


## Training: a prediction at every position

The toy model from Guide 5 shows it at small scale. Its code now lives in a shared file, `guides/tinygpt.py`, so this guide can import it instead of repeating it. The code in this guide uses PyTorch 2.14.1, pinned in the series `requirements.txt` as of October 2026, and its blocks run in order and share variables. The first block trains the toy exactly as Guide 5 did: 27 characters, 2 blocks, 112 windows of 16 characters, Adam, 500 steps. It prints the training text first, which ends "ran on the mat".

> *`guides/tinygpt.py` holds the code built in Guide 5 ([notebook 05](05-transformer.ipynb)).*

In [2]:
import sys, math
sys.path.insert(0, "guides")                   # run from the repository root
import torch
import torch.nn.functional as F
from tinygpt import TinyGPT, train, inputs, targets, vocab, text
print(text)                                    # the 128-character training text
torch.manual_seed(0)
model = TinyGPT()
final_loss = train(model, 500, report=(0, 500))   # prints the step and the training loss

the cat sat on the mat and the dog sat on the log and the cat saw the dog then the dog ran to the cat and the cat ran on the mat


0 3.306


500 0.152


One forward pass over all 112 windows gives a logit for every character at every position:

In [3]:
with torch.no_grad():
    logits = model(inputs)                     # B × n × |V|
print(tuple(logits.shape), logits.shape[0] * logits.shape[1], "predictions")
print(f"mean cross-entropy {F.cross_entropy(logits.reshape(-1, len(vocab)), targets.reshape(-1)).item():.3f}")

(112, 16, 27) 1792 predictions
mean cross-entropy 0.152


A loss in **nats**, units of the natural log that Guide 1's cross-entropy uses, is hard to picture, so it's often reported as **perplexity** (unrelated to the t-SNE setting of the same name in Guide 3), $\operatorname{PPL} = e^{\mathcal{L}}$, with $\mathcal{L}$ the average cross-entropy loss. A uniform guess over 27 characters has loss $\ln 27$ and perplexity exactly 27. So read perplexity as the number of equally likely options the distribution is effectively spread over at each step.

In [4]:
for name, loss in [("uniform guess over 27 characters", math.log(27)),
                   ("Guide 1's loss floor, one character of context", 0.884),
                   ("Guide 5's toy transformer, training text", final_loss)]:
    print(f"{name:48}loss {loss:.3f}   perplexity {math.exp(loss):.2f}")

uniform guess over 27 characters                loss 3.296   perplexity 27.00
Guide 1's loss floor, one character of context  loss 0.884   perplexity 2.42
Guide 5's toy transformer, training text        loss 0.152   perplexity 1.16


## Generation: one token at a time

Step 3, "choose one token", is the **decoding** rule (not the tokenizer's decode from Guide 2, which turns IDs back into text), and the rest of this guide is about it. Here's the toy model's distribution after "and the ", then after the 16 characters "the mat and the ", and then after each of four short contexts:

In [5]:
def probs(context, T=1.0):
    ids = torch.tensor([[vocab.index(ch) for ch in context[-16:]]]) # 1 × n character IDs, the last 16 at most
    with torch.no_grad():
        z = model(ids)[0, -1]                                        # last position's logits, |V| values
    return torch.softmax(z / T, dim=-1)                            # T is explained below
def show(p, m):                                                    # the m most probable entries
    return "  ".join(f"{vocab[i]!r} {prob:.3f}" for prob, i in zip(*p.topk(m)))
print(show(probs("and the "), 5))
print(f"{'the mat and the '!r:14}", show(probs("the mat and the "), 2))   # the full 16 characters
for context in ["the cat ", "the cat s", "the cat sa", "the cat sat"]:
    print(f"{context!r:14}", show(probs(context), 3))

'c' 0.604  'd' 0.394  'l' 0.001  'a' 0.000  ' ' 0.000
'the mat and the ' 'd' 0.998  'c' 0.001
'the cat '     's' 0.513  'r' 0.265  'a' 0.220
'the cat s'    'a' 0.999  'r' 0.000  'n' 0.000
'the cat sa'   't' 0.599  'w' 0.398  'm' 0.001
'the cat sat'  ' ' 1.000  'o' 0.000  'c' 0.000


## Choosing a token: greedy decoding

Greedy has a second failure. Continue from "the c" until the text is 220 characters long:

In [6]:
out = "the c"
while len(out) < 220:
    out += vocab[probs(out).argmax().item()]   # greedy: the most probable character
print(out)

the cat sat on the mat and the dog sat on the log and the cat saw the dog then the dog ran to the cat and the cat ran on the mat and the dog sat on the log and the cat saw the dog then the dog ran to the cat and the cat 


## Sampling and temperature

Dividing by $T < 1$ stretches the gaps between logits, so the top choice gets more of the probability. $T > 1$ shrinks the gaps and pushes toward uniform. As $T$ approaches 0 the top choice takes everything, which is greedy decoding. Recall Guide 1's softmax of $[2, 1, 0]$:

In [7]:
z = torch.tensor([2.0, 1.0, 0.0])
for T in [0.5, 1.0, 2.0]:
    print(T, [round(v, 3) for v in torch.softmax(z / T, dim=0).tolist()])

0.5 [0.867, 0.117, 0.016]
1.0 [0.665, 0.245, 0.09]
2.0 [0.506, 0.307, 0.186]


At $T = 1$ you get Guide 1's numbers back. Now the toy model after "and the ", with 1,000 draws at each temperature from one seeded random generator:

In [8]:
generator = torch.Generator().manual_seed(0)
c, d = vocab.index("c"), vocab.index("d")
for T in [0.5, 1.0, 2.0]:
    p = probs("and the ", T)
    draws = torch.bincount(torch.multinomial(p, 1000, replacement=True, generator=generator), minlength=len(vocab))
    other = 1000 - draws[c] - draws[d]
    print(f"T={T}  p(c) {p[c]:.3f}  p(d) {p[d]:.3f}  rest {1 - p[c] - p[d]:.3f}   "
          f"draws: c {draws[c]}, d {draws[d]}, other {other} over {(draws > 0).sum() - 2} characters")

T=0.5  p(c) 0.702  p(d) 0.298  rest 0.000   draws: c 707, d 293, other 0 over 0 characters
T=1.0  p(c) 0.604  p(d) 0.394  rest 0.003   draws: c 613, d 383, other 4 over 4 characters
T=2.0  p(c) 0.486  p(d) 0.392  rest 0.122   draws: c 473, d 389, other 138 over 24 characters


At $T = 2$, 138 of the 1,000 draws went to 24 other characters, including letters the model never saw after "and the ". To see the effect on text, the next block defines a `generate` function that samples one character at a time; its `k` and `p_top` arguments are for the next section.

In [9]:
def cut(p, k=None, p_top=None):                 # top-k and top-p, explained in the next section
    p, order = p.clone(), p.argsort(descending=True)
    if k is not None:
        p[order[k:]] = 0                       # keep the k most probable
    if p_top is not None:
        before = p[order].cumsum(0) - p[order] # total of the tokens ranked above each one
        p[order[before >= p_top]] = 0          # keep a token while the total before it is below p_top
    return p / p.sum()                         # renormalize
def generate(prompt, n=60, T=1.0, k=None, p_top=None, seed=0):
    generator, out = torch.Generator().manual_seed(seed), prompt
    while len(out) < n:
        p = cut(probs(out, T), k, p_top)
        out += vocab[torch.multinomial(p, 1, generator=generator).item()]
    return out
for T in [1.0, 1.5, 2.0]:
    print(T, repr(generate("the c", T=T)))

1.0 'the cat sat on the mat and the dog sat on the log and the ca'
1.5 'the cat sat on the mad rat an the d log and w the mat and th'
2.0 'the catd the dog sat oncan to the mat and thw the mat an ann'


## Cutting off the tail: top-k and top-p

Work both on five tokens with probabilities $[0.50, 0.25, 0.12, 0.08, 0.05]$. Top-2 keeps the first two, and $0.50 / 0.75$ and $0.25 / 0.75$ give $0.667$ and $0.333$. For top-p with $p_{\text{top}} = 0.9$, the running totals are 0.50, 0.75, 0.87 and 0.95. The fourth token crosses 0.9, so it's kept and the fifth is dropped. Dividing each kept value by their total, 0.95, gives 0.526, 0.263, 0.126 and 0.084. The `cut` function above does the same:

In [10]:
q = torch.tensor([0.50, 0.25, 0.12, 0.08, 0.05])
print("top-2     ", [round(v, 3) for v in cut(q, k=2).tolist()])
print("top-p 0.9 ", [round(v, 3) for v in cut(q, p_top=0.9).tolist()])

top-2      [0.667, 0.333, 0.0, 0.0, 0.0]
top-p 0.9  [0.526, 0.263, 0.126, 0.084, 0.0]


Now apply them to the toy at $T = 2$ after "and the ", 1,000 draws each:

In [11]:
generator = torch.Generator().manual_seed(0)
for name, rule in [("top-2", {"k": 2}), ("top-p 0.9", {"p_top": 0.9})]:
    p = cut(probs("and the ", 2.0), **rule)
    draws = torch.bincount(torch.multinomial(p, 1000, replacement=True, generator=generator), minlength=len(vocab))
    print(f"{name:10}", {vocab[i]: draws[i].item() for i in range(len(vocab)) if draws[i] > 0})

top-2      {'c': 559, 'd': 441}
top-p 0.9  {'a': 16, 'c': 544, 'd': 417, 'l': 23}


## The running example

Back to "The cat sat on the" (token IDs 464, 3797, 3332, 319, 262) and GPT-2 small. Its weights are a 548 MB download, too much to run in every reader's session, so this block is shown but not run here. The functions live in `series/running-example.py`, which pins the model revision, checks the file's hash and writes out GPT-2's forward pass in a few lines of PyTorch.

In [12]:
%%gpt2
import runpy
script = runpy.run_path("series/running-example.py")         # needs the 548 MB weights file
for T in [0.5, 1.0, 2.0]:
    print(T, script["top_k"](5, T)[0])                         # the 5 most probable next tokens at T
share, rank, p_mat, kept = script["tail_stats"]()                  # at T = 1
print("top-5 share", share, "| \" mat\" rank", rank, "p", p_mat, "| top-p keeps", kept)

Skipped: needs GPT-2 weights (see the setup cell to enable).


---

← [Guide 5: The transformer](05-transformer.ipynb)

Found a cell that doesn't reproduce the guide? [Open an issue](https://github.com/drvgautam/primer-on-aiengineering/issues/new?template=does-not-reproduce.yml).